# Fruit Ripeness Classification

## Import libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Load Dataset

In [5]:
# Load CSVs
train_df = pd.read_csv('train_labels.csv')
test_df = pd.read_csv('test_labels.csv')

# Optional: Convert numeric class to string label for clarity
label_map = {
    0: 'Raw_Banana',
    1: 'Raw_Mango',
    2: 'Ripe_Banana',
    3: 'Ripe_Mango'
}
train_df['class'] = train_df['class'].map(label_map)
test_df['class'] = test_df['class'].map(label_map)

# Define image generator (no split)
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='data/train/images',
    x_col='filename',
    y_col='class',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

test_gen = datagen.flow_from_dataframe(
    dataframe=test_df,
    directory='data/test/images',
    x_col='filename',
    y_col='class',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 3999 validated image filenames belonging to 4 classes.
Found 1000 validated image filenames belonging to 4 classes.


In [24]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

base_model = MobileNetV2(include_top=False, input_shape=(224,224,3), weights='imagenet')
base_model.trainable = False  # or True for fine-tuning

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [25]:
model.fit(train_gen, epochs=10, validation_data=test_gen)

C:\Users\Kenny\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 262s 2s/step - accuracy: 0.7538 - loss: 0.6174 - val_accuracy: 0.9940 - val_loss: 0.0515
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 121s 963ms/step - accuracy: 0.9709 - loss: 0.0877 - val_accuracy: 0.9870 - val_loss: 0.0422
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - accuracy: 0.9762 - loss: 0.0667 - val_accuracy: 0.9770 - val_loss: 0.0538
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 118s 939ms/step - accuracy: 0.9849 - loss: 0.0458 - val_accuracy: 0.9850 - val_loss: 0.0467
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 118s 941ms/step - accuracy: 0.9877 - loss: 0.0355 - val_accuracy: 0.9930 - val_loss: 0.0166
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 118s 941ms/step - accuracy: 0.9895 - loss: 0.0306 - val_accuracy: 0.9900 - val_loss: 0.0233
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 118s 941ms/step - accuracy: 0.9942 - loss: 0.0181 - val_accuracy: 0.9860 - val_loss: 0.0354
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 143s 946ms/step - accuracy: 0.9914 - loss:

In [46]:
import numpy as np
from tensorflow.keras.preprocessing import image
def predict_img(image_path):
    test_image = image.load_img(path=image_path, target_size=(224,224))
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis=0)
    class_names = ['Raw_Banana', 'Raw_Mango', 'Ripe_Banana','Ripe_Mango']
    result = model.predict(test_image)
    predicted_class = class_names[np.argmax(result)]
    confidence = np.max(result)

    print(f"Predicted class: {predicted_class} ({confidence:.2f} confidence)")

In [28]:
model.save('ripeness_classification.h5', include_optimizer=True)

In [29]:
model.save('ripeness_classification.keras')